# Binary Search Tree Optimisation

How does choosing a distribution over BST roots change the **worst expected reward** across keys? This is a deterministic toy allocation model. A root is a key, not a random-number seed.

The original exploratory notebook is retained in Git history. This revised walkthrough uses tested functions and a linear programme with a precisely stated objective.

## Model

Keys are the integers 1–100. The root candidates are 38–62. Subsequent splits choose the middle key furthest from its parent, with ties going to the upper middle. A key at depth d receives reward 6−d. The zero threshold is explicit. Changing these assumptions defines a different optimisation problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bst_rewards import tree_depths, reward_matrix, optimise_mixture

roots = np.arange(38, 63)
rewards = reward_matrix(roots, n=100)
uniform = rewards.mean(axis=0)
print("Baseline minimum:", uniform.min())
print("Baseline keys below zero:", np.count_nonzero(uniform < -1e-9))

## Optimisation

Choose nonnegative root weights w summing to one. Maximise t subject to every expected key reward being at least t: `R.T @ w >= t`. This is a linear programme over a fixed set of candidate trees. It does not optimise all possible tree shapes.

In [ ]:
result = optimise_mixture(rewards)
print("Maximin minimum:", result.minimum_reward)
print("Maximin keys below zero:", np.count_nonzero(result.expected_rewards < -1e-9))
print("Weight sum:", result.weights.sum())
print("Mean reward, uniform / maximin:", uniform.mean(), result.expected_rewards.mean())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
keys = np.arange(1, 101)
ax.plot(keys, uniform, ".-", label="Uniform mixture", linewidth=.6)
ax.plot(keys, result.expected_rewards, ".-", label="Maximin mixture", linewidth=.6)
ax.axhline(0, color="grey", linewidth=.8)
ax.set(xlabel="Key", ylabel="Expected reward", title="Worst-case reward and threshold-count trade-off")
ax.legend()
plt.show()

## Interpretation

The minimum expected reward improves from −0.5600 to approximately −0.1982. The number of keys below zero increases from 13 to 38, while the mean stays at 0.2. Improving the worst outcome and reducing the number of negative outcomes are different objectives.

This result is the numerical maximin optimum for the specified candidate matrix. It is not a universal BST theorem or an improvement to lookup complexity. The old README counts of 27 and 16 are not reproduced by this explicit baseline, so they are not retained as current benchmark claims.

## Try another well-defined scenario

Change the candidate set while keeping the reward rule fixed. Comparing the resulting lower envelope shows how much of the result depends on the permitted roots.

In [ ]:
all_roots = reward_matrix(range(1, 101), n=100)
expanded = optimise_mixture(all_roots)
print("Minimum reward with all 100 roots:", expanded.minimum_reward)
assert expanded.minimum_reward >= result.minimum_reward - 1e-8

With all 100 root candidates available, the minimum expected reward becomes approximately **+0.0285**, so every key has positive expected reward. The mean falls to approximately **0.1057**. This is a trade-off between protecting the worst outcome and preserving the average.

The guarantee is an expectation over the chosen root distribution; an individual tree still has keys with negative rewards. The CSV files produced by `python demo.py` provide the full weight vector and per-key expectations.